<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/structured_document_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain langchain-openai pydantic pypdf markdown --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 20.7 MB/s eta 0:00:00


In [5]:
from pathlib import Path

from pydantic import BaseModel
from pypdf import PdfReader
import markdown

from langchain_openai import ChatOpenAI
from google.colab import userdata

In [7]:
OPENAI_MODEL_NAME=userdata.get('OPENAI_MODEL_NAME')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [10]:
PROMPT = f"""
Read the text below, extract the installation-related information and organize it into three sections:

1. prerequisites
2. installation
3. post_install_instructions

Text:

"""

In [8]:
class InstructionSections(BaseModel):
    prerequisites: list[str]
    installation: list[str]
    post_install_instructions: list[str]

In [11]:
def organize_text_into_sections(text):
    llm = ChatOpenAI(api_key=OPENAI_API_KEY,model=OPENAI_MODEL_NAME)

    structured_llm = llm.with_structured_output(
        InstructionSections,
        method="json_schema"
    )

    response=structured_llm.invoke(PROMPT+text)

    return response

In [15]:
def extract_text_from_text_file(file_path):
  with open(file_path, "r", encoding="latin-1") as file:
      raw_text = file.read()
  return raw_text


def extract_text_from_pdf_with_pypdf(file_path):
    reader = PdfReader(file_path)

    text_parts = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_parts.append(page_text)

    return "\n\n".join(text_parts)


def extract_text(file_path, pdf_method="pypdf"):
    path = Path(file_path)
    file_type = path.suffix.lower()

    if file_type in [".md", ".txt"]:
        return extract_text_from_text_file(file_path)

    if file_type == ".pdf":
        if pdf_method == "pypdf":
            return extract_text_from_pdf_with_pypdf(file_path)

    raise ValueError("Please use a .txt, .md, or .pdf file.")

In [16]:
file_path = "/content/trane_file_G2.txt"

raw_text = extract_text(file_path, pdf_method="pypdf")
sections = organize_text_into_sections(raw_text)

In [17]:
# sections = organize_text_into_sections(raw_text)

sections_as_dict = sections.model_dump()
sections_as_json = sections.model_dump_json(indent=2)

print(sections_as_json)

{
  "prerequisites": [
    "Installation and maintenance must be performed by qualified/licensed personnel with appropriate electrical, electronic and mechanical experience.",
    "Comply with all applicable national and local electrical codes and site regulations.",
    "Cut power and discharge capacitors before any electrical or refrigerant work; use an all-pole isolating air switch with ≥3 mm contact separation when required.",
    "Ensure reliable earth/ground connection; grounding conductor must not be used for other purposes.",
    "Do not share the air conditioner circuit with other appliances; use a dedicated circuit sized per the unit nameplate.",
    "Handle units >20 kg with two or more people and wear a safety belt when working above 2 m.",
    "For R32/R290 refrigerants: follow combustible refrigerant rules (room-area/LFL calculations), install only in suitably ventilated locations, avoid open flames/hot work, and use anti-static clothing and gloves.",
    "Have the follow

In [19]:
def make_markdown(sections):
    markdown_text = "# Setup Instructions\n\n"

    markdown_text += "## Prerequisites\n\n"
    for item in sections.prerequisites:
        markdown_text += f"- {item}\n"

    markdown_text += "\n## Installation\n\n"
    for item in sections.installation:
        markdown_text += f"- {item}\n"

    markdown_text += "\n## Post-install Instructions\n\n"
    for item in sections.post_install_instructions:
        markdown_text += f"- {item}\n"

    return markdown_text


def make_html(sections):
    markdown_text = make_markdown(sections)
    html_text = markdown.markdown(markdown_text)
    return html_text

In [21]:
markdown_output = make_markdown(sections)
html_output = make_html(sections)

with open("setup_instructions.md", "w", encoding="utf-8") as file:
    file.write(markdown_output)

full_html_output = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Setup Instructions</title>
</head>
<body>
{html_output}
</body>
</html>
"""

with open("setup_instructions.html", "w", encoding="utf-8") as file:
    file.write(full_html_output)

In [22]:
from IPython.display import Markdown, HTML, display

In [23]:
display(Markdown(markdown_output))

# Setup Instructions

## Prerequisites

- Installation and maintenance must be performed by qualified/licensed personnel with appropriate electrical, electronic and mechanical experience.
- Comply with all applicable national and local electrical codes and site regulations.
- Cut power and discharge capacitors before any electrical or refrigerant work; use an all-pole isolating air switch with ≥3 mm contact separation when required.
- Ensure reliable earth/ground connection; grounding conductor must not be used for other purposes.
- Do not share the air conditioner circuit with other appliances; use a dedicated circuit sized per the unit nameplate.
- Handle units >20 kg with two or more people and wear a safety belt when working above 2 m.
- For R32/R290 refrigerants: follow combustible refrigerant rules (room-area/LFL calculations), install only in suitably ventilated locations, avoid open flames/hot work, and use anti-static clothing and gloves.
- Have the following calibrated/appropriate tools and safety equipment: explosion-proof vacuum pump (vacuum <10 Pa), explosion-proof filling device, leak detector, combustible refrigerant concentration detector (fixed and portable), calibrated pressure gauges, and suitable fire extinguishers (dry powder, CO2, foam).
- Ensure maintenance/installation area is well ventilated, free of ignition sources (welding, smoking, ovens), and meets minimum room-area requirements for refrigerant charge.
- Verify warning labels are intact and visible; replace damaged labels before energizing.
- Confirm the required pipe sizes, maximum piping lengths, and refrigerant charge (refer to nameplate and appendix).
- Keep appropriate spare parts and use only manufacturer-recommended components and consumables (insulation, tape, flaring tools).

## Installation

- Select indoor and outdoor mounting locations per manual: strong level wall, unobstructed air inlet/outlet, easy piping/routing, easy condensate drainage, and minimum clearances shown in diagrams; install indoor unit ≥2.5 m above ground if specified.
- Avoid locations near heat, steam, corrosive/flammable gases, excessive dust, or where noise or discharge will disturb neighbors.
- Use the supplied accessories (mounting plate, remote, drain hose, hole cover, screws, insulation, vinyl tape, batteries) and manufacturer-appointed components only.
- Mounting plate installation: position with a level, drill holes ~32 mm deep, insert plastic anchors, fix with supplied tapping screws and verify plate is secure.
- Drill wall piping hole with a downward slope toward exterior; fit a flexible flange; keep drain pipe sloped down to exterior to prevent leaks.
- Electrical wiring: follow safety rules, use qualified circuit and air switch, ensure power supply matches unit specification. Select power cable size per appliance amperage (refer to wire size table).
- Indoor electrical connections: open front panel, access terminal block, connect wires to numbered terminals using appropriate wire size, ensure outdoor cable is suitable for outdoor use, ensure plug remains accessible, and maintain a good earth connection.
- Outdoor electrical connections: remove cover, connect cable wires to terminal board using same numbering as indoor unit, fasten cables with clamps, restore covers.
- Refrigerant piping routing: run piping in permitted directions, cut notches in IDU housing when required, bind copper pipes, drain hose and power cable together with drain at bottom, avoid bending pipe >3 times at one point.
- Pipe connection procedure: keep caps on until connection, flare properly per flaring guidelines, remove burrs before flaring, use two wrenches when tightening (opposite directions), wrap and overlap insulation, secure with vinyl tape. Follow recommended flare dimensions and re-cut/re-flare if defective.
- Pipe sizes and tightening torques: use correct gas/liquid pipe diameters and torques per model (e.g., 7/9/12K gas 3/8" torque 4.2 kg·m; 18K gas 1/2" torque 5.5 kg·m; 24K gas 5/8" torque 6.6 kg·m; liquid 1/4" torque 1.8 kg·m for many models).
- Outdoor unit mounting: install on solid support, use abundant screw anchors appropriate for wall type, use rubber gaskets on feet if vibration present, leave service clearances and fasten securely to avoid movement.
- Heat-pump ODU condensate: for heat pump models, install drain port in specified 25 mm hole and route drain pipe to suitable discharge location.
- Vacuum and leak test: evacuate refrigerant circuit with vacuum pump (bleed air and moisture); operate piezometer 10–15 minutes and verify pressure ~-0.1 MPa; close pump and hold 1–2 minutes to check for pressure rise; if pressure falls, check for leaks.
- After vacuum, open liquid and gas valve cores fully, tighten valve caps and refrigerant charging vent screws.
- Leak detection: check all joints with calibrated leak detector; if detector unavailable, use soap-water method and inspect for bubbles ≥3 minutes.
- Final test and commissioning: obtain client approval of installation, explain important notes, energize power, start unit via remote (ON/OFF), cycle through modes (AUTO/COOL/DRY/FAN/HEAT) to verify operation; note that cooling may not start if ambient <16 °C.
- Bundle piping and drain hose where they enter rear of indoor unit, insert and secure pipe connection into slot, and press to join. Ensure insulation has no gaps and wrap rear piping housing with vinyl tape.
- Follow required safety procedures when any welding is necessary: drain and purge refrigerant, ensure no refrigerant in system, close outdoor stop valve, and perform welding only when safe.
- Use nitrogen to purge the system before operating and vacuumize outdoor unit for at least 30 minutes when filling refrigerant or after brazing/welding.

## Post-install Instructions

- Perform a full leak test after installation using the calibrated leak detector; do not energize the system if leaks are detected until repaired.
- Fill refrigerant strictly per nameplate type and volume; avoid overfilling; if additional charge required use manufacturer formula (appendix) and respect maximum piping length limits.
- Seal the refrigerant system safely after work and ensure insulation and protective covers are properly replaced.
- Verify and demonstrate to the client how to operate the unit and remote control functions, timer and emergency switch; point out any special restrictions (e.g., cooling may not start below 16 °C).
- Provide client with important safety notes and maintenance recommendations and obtain client approval of installation before commissioning.
- Record installation parameters (refrigerant charge, pipe length, pressures, vacuum hold readings) and leave labeling/ warning stickers intact and visible.
- Advise the client about emergency and leak procedures: evacuate area, ventilate, shut off power, inform neighbors if large leak, and contact professional service or emergency responders as needed.
- If refrigerant recovery is required (relocation/repair), run unit in cooling mode and close high-pressure valve then low-pressure valve after 30–40 s, stop unit and disconnect power; do not exceed the recommended recovery time.
- Store and label refrigerant cylinders upright in ventilated area (−10 °C to 50 °C) and ensure cylinders do not exceed stipulated filled volume when transported or kept on site.
- Perform a safety inspection and leak check before any future maintenance or repair on units with combustible refrigerant; if system is opened for major repair, return product to authorized maintenance station for welding and major brazing.
- After finishing installation and commissioning, ensure the system is grounded, warning labels are present, valve caps tightened, and that the system has passed the vacuum/hold and leakage checks.
- If any refrigerant leak or damage occurs during or after installation, do not perform on-site welding on the damaged product; return it to a service center and follow emergency handling procedures.


In [24]:
display(HTML(html_output))